> **Note:** This notebook requires a live OpenAI API key. Set `OPENAI_API_KEY` in your `.env` file before running. Smoke-run deferred — API key not available in CI.

# 1. Model - abstracts over the LLM API
# 2. Prompt Template - abstracts over the Prompts sent to the LLMs
# 3. Output Parser - transforms raw output into workable formats (strings, jsons, etc...)

# Document Loaders, Retrievers, Vector Stores, Agents, Tools, ......

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.4-mini", use_responses_api=True)
llm.invoke("What is LangChain?")

In [ ]:
output = llm.invoke("What is LangChain?")

output

In [ ]:
type(output)

In [ ]:
output.response_metadata

In [ ]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

output_parser.invoke(output)

# LCEL! - LangChain Expression Language
It is a special language to combine langchain components (lego pieces!) into CHAINS! (REUSABLE BUILDING BLOCKS)

In [ ]:
chain = llm | output_parser

chain.invoke("What is LangChain?")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("What is {ai_python_framework}")

chain = prompt | llm | output_parser

chain.invoke({"ai_python_framework": "LangChain"})

In [ ]:
chain.invoke({"ai_python_framework": "Pydantic?"})

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2")

chain = prompt | llm | output_parser

print(chain.invoke({"ai_python_framework": "LangChain"}))
print(chain.invoke({"ai_python_framework": "Pydantic?"}))

In [ ]:
def reusable_chain(llm):
    chain = prompt | llm | output_parser
    return chain

llm_openai = ChatOpenAI(model="gpt-5.4-mini", use_responses_api=True)
llm_ollama = ChatOllama(model="llama3.2")

chain1 = reusable_chain(llm_openai)
chain2 = reusable_chain(llm_ollama)

chain1.invoke({"ai_python_framework": "LangChain"})
chain2.invoke({"ai_python_framework": "Pydantic?"})

# Processing PDFs and organizing them into a Table 

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("./assets-resources/attention-paper.pdf")

docs = loader.load()

docs[0]

In [ ]:
from IPython.display import Markdown

Markdown(str(docs[0]))

In [ ]:
docs[0].metadata

In [ ]:
def load_docs_to_string(docs):
    return "\n".join([doc.page_content for doc in docs])

docs_string = load_docs_to_string(docs)

docs_string

In [ ]:
llm = ChatOpenAI(model="gpt-5.4-mini", use_responses_api=True)

In [ ]:
def pdf_summarizer(docs_string):
    prompt = ChatPromptTemplate.from_template("Summarize the following document: {document} as bullet points:")
    chain_summarizer = prompt | llm | output_parser
    return chain_summarizer.invoke({"document": docs_string})


summary_pdf = pdf_summarizer(docs_string)

Markdown(summary_pdf)

In [ ]:
import pandas as pd

def update_summary_dataframe(docs, summary, df=None):
    """
    Updates a pandas DataFrame with document metadata and summary.
    Creates a new DataFrame if none is provided.
    
    Args:
        docs: List of documents with metadata
        summary: Summary text of the documents
        df: Optional existing DataFrame to update
        
    Returns:
        Updated pandas DataFrame
    """
    # Create a dictionary with the new data
    new_data = {
        'paper_path': [docs[0].metadata['source']],
        'summary': [summary]
    }
    
    # Create new DataFrame with the data
    new_df = pd.DataFrame(new_data)
    
    # If existing df provided, concatenate with new data
    if df is not None:
        return pd.concat([df, new_df], ignore_index=True)
    
    return new_df

# Create initial DataFrame
df = update_summary_dataframe(docs, summary_pdf)

# Display the DataFrame 
df

In [ ]:
pdf_path2 = "./assets-resources/llm_paper_know_dont_know.pdf"

loader2 = PyPDFLoader(pdf_path2)

docs2 = loader2.load()

docs2[0]

docs2_string = load_docs_to_string(docs2)

docs2_string

summary_pdf2 = pdf_summarizer(docs2_string)

Markdown(summary_pdf2)

In [ ]:
df2 = update_summary_dataframe(docs2, summary_pdf2, df)

df2